In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import nltk
import re
import matplotlib.pyplot as plt
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import KFold

In [3]:
news_df = pd.read_csv('/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv')
df = news_df
news_df.head()

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83


In [4]:
a = df[df["true_sentiment"] == "Negative"]["text"][8] # observing how a negative looks like

In [5]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

Create a bag of words dictionary for each class and in total. For creating the file, we will go through each "document" and add  all of our data to the bag of words dictionary. We will ignore all of the words which will have only special characters (dot, comma, paranthesis etc.), and also numbers due to not actually helping a lot in terms of sentiment analysis.

In [14]:
def create_bag_of_words(df, label):
  bag_dict = {
      "Positive": {},
      "Positive_count": 0,
      "Positive_count_frequency": 0,
      "Neutral": {},
      "Neutral_count": 0,
      "Neutral_count_frequency": 0,
      "Negative": {},
      "Negative_count": 0,
      "Negative_count_frequency": 0,
      "total": {},
      "total_count_frequency": 0,
      "total_count": len(df)
  }

  regex_filter = "[A-Za-z]+"

  for i in df.index:
    bag_dict[df["true_sentiment"][i] + '_count'] += 1
    doc_tokenized = [el.lower() for el in nltk.word_tokenize(df[label][i]) if re.search(regex_filter, el)]
    for el in doc_tokenized:
      if el not in bag_dict[df["true_sentiment"][i]]:
        bag_dict[df["true_sentiment"][i]][el] = 1
      else:
        bag_dict[df["true_sentiment"][i]][el] += 1
      if el not in bag_dict["total"]:
        bag_dict["total"][el] = 1
      else:
        bag_dict["total"][el] += 1
      bag_dict[df["true_sentiment"][i] + '_count_frequency'] += 1
      bag_dict["total_count_frequency"] += 1

  return bag_dict

Separate into two datasets: train and test, and after that get your bag of words dictionaries.

In [29]:
df_train, df_test = train_test_split(df, test_size=0.2)
label = "text"
text_dict = create_bag_of_words(df_train, label)

Now, let's validate the test data.
We will make a function for testing our results with the dictionary created before.

In [26]:
def get_predictions(doc_dict, df):
  prediction = []
  regex_filter = "[A-Za-z]+"
  classes = ["Positive", "Neutral", "Negative"]
  for i in df.index:

      doc_tokenized = [el.lower() for el in nltk.word_tokenize(df[label][i]) if re.search(regex_filter, el)]

      results = {}
      V_len = len(doc_dict["total"].keys())
      for cls in classes:
        results[cls] = np.log(doc_dict[cls + "_count"] / doc_dict["total_count"])
        cls_freq = doc_dict[cls + "_count_frequency"]
        for el in doc_tokenized:
          # we apply LaPlace (adding + 1 to each word)
          word_count = doc_dict[cls][el] + 1 if el in doc_dict[cls] else 1
          results[cls] += np.log(word_count / (cls_freq + V_len))

      prediction.append(max(results, key=results.get))

  return prediction

In [30]:
predictions = get_predictions(text_dict, df_test)


Now, we will create a confusion matrix, and we will see the results of different metrics: (accuracy, precision, recall, F1-score).

In [31]:
print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

              precision    recall  f1-score   support

    Positive       0.48      0.37      0.42       155
     Neutral       0.47      0.55      0.51       161
    Negative       0.42      0.44      0.43       143

    accuracy                           0.46       459
   macro avg       0.46      0.46      0.45       459
weighted avg       0.46      0.46      0.45       459



Now, let's test it from other perspectives:

In [33]:
df_train, df_test = train_test_split(df, test_size=0.2)
label = "title"
text_dict = create_bag_of_words(df_train, label)

predictions = get_predictions(text_dict, df_test)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

              precision    recall  f1-score   support

    Positive       0.67      0.70      0.68       160
     Neutral       0.64      0.59      0.61       156
    Negative       0.64      0.66      0.65       143

    accuracy                           0.65       459
   macro avg       0.65      0.65      0.65       459
weighted avg       0.65      0.65      0.65       459



Some words are better fitted in terms of a positive sentiment when it comes only to the occurence of the word. We will calculate only the occurences and apply the same tests as before.

In [34]:
def create_bag_of_words_occurences(df, label):
  bag_dict = {
      "Positive": {},
      "Positive_count": 0,
      "Positive_count_frequency": 0,
      "Neutral": {},
      "Neutral_count": 0,
      "Neutral_count_frequency": 0,
      "Negative": {},
      "Negative_count": 0,
      "Negative_count_frequency": 0,
      "total": {},
      "total_count_frequency": 0,
      "total_count": len(df)
  }

  regex_filter = "[A-Za-z]+"

  for i in df.index:

    bag_dict[df["true_sentiment"][i] + '_count'] += 1
    doc_tokenized = [el.lower() for el in nltk.word_tokenize(df[label][i]) if re.search(regex_filter, el)]
    occurence_words = {}

    for el in doc_tokenized:
      if el not in occurence_words:
        occurence_words[el] = 0
        if el not in bag_dict[df["true_sentiment"][i]]:
          bag_dict[df["true_sentiment"][i]][el] = 1
        else:
          bag_dict[df["true_sentiment"][i]][el] += 1
        if el not in bag_dict["total"]:
          bag_dict["total"][el] = 1
        else:
          bag_dict["total"][el] += 1
        bag_dict[df["true_sentiment"][i] + '_count_frequency'] += 1
        bag_dict["total_count_frequency"] += 1

  return bag_dict

Now let's train and test it:


In [35]:
df_train, df_test = train_test_split(df, test_size=0.2)
label = "text"
text_dict = create_bag_of_words_occurences(df_train, label)

predictions = get_predictions(text_dict, df_test)
print("Occurences metrics for the text:")
print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

df_train, df_test = train_test_split(df, test_size=0.2)
label = "title"
print("Occurences metrics for the title:")
text_dict = create_bag_of_words_occurences(df_train, label)

predictions = get_predictions(text_dict, df_test)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Occurences metrics for the text:
              precision    recall  f1-score   support

    Positive       0.44      0.50      0.47       146
     Neutral       0.53      0.50      0.51       171
    Negative       0.47      0.44      0.45       142

    accuracy                           0.48       459
   macro avg       0.48      0.48      0.48       459
weighted avg       0.48      0.48      0.48       459

Occurences metrics for the title:
              precision    recall  f1-score   support

    Positive       0.61      0.70      0.65       153
     Neutral       0.69      0.58      0.63       168
    Negative       0.59      0.62      0.60       138

    accuracy                           0.63       459
   macro avg       0.63      0.63      0.63       459
weighted avg       0.63      0.63      0.63       459



Let's try a K-fold validation by the text and by the title:

In [ ]:
kf = KFold(n_splits=5)
kf.get_n_splits(df)

5

In [ ]:
for i, (train_index, test_index) in enumerate(kf.split(df)):
  df_train = df.iloc[train_index]
  df_test = df.iloc[test_index]
  print("Kfold {}".format(i))
  print("For text")
  label = "text"
  text_dict = create_bag_of_words_occurences(df_train, label)
  predictions = get_predictions(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))
  print("For title")
  label = "title"
  text_dict = create_bag_of_words_occurences(df_train, label)
  predictions = get_predictions(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold 0
For text
              precision    recall  f1-score   support

    Positive       0.39      0.39      0.39       135
     Neutral       0.41      0.53      0.47       150
    Negative       0.47      0.36      0.41       174

    accuracy                           0.42       459
   macro avg       0.42      0.42      0.42       459
weighted avg       0.43      0.42      0.42       459

For title
              precision    recall  f1-score   support

    Positive       0.56      0.65      0.60       135
     Neutral       0.57      0.49      0.53       150
    Negative       0.60      0.60      0.60       174

    accuracy                           0.58       459
   macro avg       0.58      0.58      0.58       459
weighted avg       0.58      0.58      0.58       459

Kfold 1
For text
              precision    recall  f1-score   support

    Positive       0.36      0.50      0.41       139
     Neutral       0.46      0.48      0.47       156
    Negative       0.51      0.

The article of the dataset says that the data is split in 6 key pair categories which are relevant: AUDUSD, EURCHF, EURUSD, GBPUSD, and USDJPY. We will apply naive bayes with this variable in addition as a prior knowledge, with the assumption that it is independent with the data and classes. "t" will be the pair class.

$$P(c|d,t) ∝ P(d,t|c)*P(c) = P(d|c)*P(t|c)*P(c) $$



In [44]:
def create_bag_of_words_occurences_ticker(df, label):
  bag_dict = {
      "Positive": {},
      "Positive_count": 0,
      "Positive_count_frequency": 0,
      "Neutral": {},
      "Neutral_count": 0,
      "Neutral_count_frequency": 0,
      "Negative": {},
      "Negative_count": 0,
      "Negative_count_frequency": 0,
      "total": {},
      "total_count_frequency": 0,
      "total_count": len(df)
  }

  regex_filter = "[A-Za-z]+"

  for i in df.index:

    bag_dict[df["true_sentiment"][i] + '_count'] += 1
    doc_tokenized = [el.lower() for el in nltk.word_tokenize(df[label][i]) if re.search(regex_filter, el)]
    occurence_words = {}
    ticker = df["ticker"][i]
    if ticker not in bag_dict[df["true_sentiment"][i]]:
      bag_dict[df["true_sentiment"][i]][ticker] = 1
    else:
      bag_dict[df["true_sentiment"][i]][ticker] += 1

    if ticker not in bag_dict["total"]:
      bag_dict["total"][ticker] = 1
    else:
      bag_dict["total"][ticker] += 1

    for el in doc_tokenized:
      if el not in occurence_words:
        occurence_words[el] = 0
        if el not in bag_dict[df["true_sentiment"][i]]:
          bag_dict[df["true_sentiment"][i]][el] = 1
        else:
          bag_dict[df["true_sentiment"][i]][el] += 1
        if el not in bag_dict["total"]:
          bag_dict["total"][el] = 1
        else:
          bag_dict["total"][el] += 1
        bag_dict[df["true_sentiment"][i] + '_count_frequency'] += 1
        bag_dict["total_count_frequency"] += 1

  return bag_dict

def get_predictions_ticker(doc_dict, df):
  prediction = []
  regex_filter = "[A-Za-z]+"
  classes = ["Positive", "Neutral", "Negative"]
  for i in df.index:

      doc_tokenized = [el.lower() for el in nltk.word_tokenize(df[label][i]) if re.search(regex_filter, el)]

      results = {}
      V_len = len(doc_dict["total"].keys())
      for cls in classes:
        results[cls] =0
        results[cls] += doc_dict[cls][df["ticker"][i]] / doc_dict["total"][df["ticker"][i]]
        cls_freq = doc_dict[cls + "_count_frequency"]
        for el in doc_tokenized:
          # we apply LaPlace (adding + 1 to each word)
          word_count = doc_dict[cls][el] + 1 if el in doc_dict[cls] else 1
          results[cls] += word_count / (cls_freq + V_len)

      print(results)
      prediction.append(max(results, key=results.get))

  return prediction

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2)
label = "text"
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker(text_dict, df_test)
print("Occurences metrics for the text:")
print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

df_train, df_test = train_test_split(df, test_size=0.2)
label = "title"
print("Occurences metrics for the title:")
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker(text_dict, df_test)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold for the last one

In [ ]:
for i, (train_index, test_index) in enumerate(kf.split(df)):
  df_train = df.iloc[train_index]
  df_test = df.iloc[test_index]
  print("Kfold {}".format(i))
  print("For text")
  label = "text"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))
  print("For title")
  label = "title"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold 0
For text
              precision    recall  f1-score   support

    Positive       0.39      0.39      0.39       135
     Neutral       0.41      0.53      0.46       150
    Negative       0.45      0.34      0.39       174

    accuracy                           0.41       459
   macro avg       0.42      0.42      0.41       459
weighted avg       0.42      0.41      0.41       459

For title
              precision    recall  f1-score   support

    Positive       0.54      0.64      0.59       135
     Neutral       0.55      0.50      0.52       150
    Negative       0.59      0.55      0.57       174

    accuracy                           0.56       459
   macro avg       0.56      0.57      0.56       459
weighted avg       0.56      0.56      0.56       459

Kfold 1
For text
              precision    recall  f1-score   support

    Positive       0.36      0.49      0.41       139
     Neutral       0.46      0.49      0.47       156
    Negative       0.51      0.